# Introduction to Transfer Learning

Transfer learning means using knowledge learned from one task to improve learning on another related task.

In computer vision, this usually means starting from a CNN trained on a large image dataset and adapting it to a smaller custom dataset.

## Learning Goals

- Understand why transfer learning works.
- Learn feature extraction and fine-tuning.
- Understand freezing, unfreezing, and replacing classifier heads.
- Learn common transfer learning workflows.
- Understand U-Net and image segmentation conceptually.
- Learn caution points for real projects.

> **Scope:** This is a theory notebook. Code is kept in `03_introduction_To_Transfer_learning_add_code.ipynb`.


# Why Transfer Learning?

Training a large CNN from scratch can require:

- Huge datasets
- Long training time
- Large compute resources
- Careful tuning

Transfer learning avoids starting from zero.

## Core Idea

A CNN trained on a large dataset learns useful visual features.

Early layers often learn:

- Edges
- Corners
- Color blobs
- Simple textures

Middle layers may learn:

- Parts
- Repeated shapes
- Object-like patterns

Later layers become more specific to the original training task.

## Example

A model trained on a large general image dataset may already understand edges, textures, and shapes. If your task is classifying pizza, steak, and sushi, you can reuse those learned features and train only a small classifier.

## Key Points

- Transfer learning reduces data requirements.
- It can improve accuracy on small datasets.
- It speeds up training.
- It works best when source and target domains are related.

## Caution Points

- Transfer learning is not magic.
- If the new data is very different, pretrained features may transfer poorly.
- The final classifier must match your number of classes.


# Feature Extraction vs Fine-Tuning

There are two common transfer learning strategies.

## 1. Feature Extraction

In feature extraction:

- Keep the pretrained backbone frozen.
- Replace the classifier head.
- Train only the new head.

This is useful when:

- Dataset is small.
- Compute is limited.
- The new task is similar to the original task.

## 2. Fine-Tuning

In fine-tuning:

- Start with pretrained weights.
- Replace the classifier head.
- Unfreeze some deeper layers.
- Train those layers with a small learning rate.

This is useful when:

- Dataset is moderately large.
- The new task differs from the original task.
- You need higher performance.

## Comparison

| Strategy | Frozen Backbone? | Trainable Layers | Risk of Overfitting | Compute Need |
|---|---|---|---|---|
| Feature extraction | Yes | Classifier head | Lower | Lower |
| Fine-tuning | Partly or no | Head + selected backbone layers | Higher | Higher |

## Key Points

- Feature extraction is safer for small datasets.
- Fine-tuning can improve performance when enough data is available.
- Fine-tuning should usually use a smaller learning rate for pretrained layers.

## Caution Points

- Unfreezing too much too early can destroy useful pretrained features.
- Training all layers on a tiny dataset can overfit quickly.
- Always compare validation performance.


# Anatomy of a Pretrained CNN

A pretrained CNN usually has two major parts:

1. **Backbone / Feature Extractor**
2. **Classifier Head**

## Backbone

The backbone extracts visual features from the image.

Examples:

- ResNet
- VGG
- EfficientNet
- MobileNet

## Classifier Head

The classifier head maps features to output classes.

If the pretrained model was trained on 1000 classes, its original head outputs 1000 values.

For a new task with 3 classes, replace the head so it outputs 3 values.

## Example

Original model:

$$\text{Image} \rightarrow \text{Backbone} \rightarrow 1000 \text{ ImageNet classes}$$

New model:

$$\text{Image} \rightarrow \text{Same Backbone} \rightarrow 3 \text{ food classes}$$

## Key Points

- The backbone is reused.
- The head is task-specific.
- The output layer must match the new task.

## Caution Points

- Do not forget to replace the final layer.
- Check input image size expected by the model.
- Use the normalization expected by the pretrained model.


# The Transfer Learning Workflow

## Step 1: Choose a Pretrained Model

Choose a model based on:

- Accuracy need
- Speed requirement
- Memory limit
- Dataset size

## Step 2: Prepare Data

Images usually need:

- Resize
- Crop
- Tensor conversion
- Normalization
- Augmentation

## Step 3: Freeze the Backbone

Freezing means preventing pretrained weights from being updated.

This keeps learned general visual features intact.

## Step 4: Replace the Classifier Head

The new head must match your target classes.

Example:

For pizza, steak, and sushi:

$$\text{output neurons} = 3$$

## Step 5: Train the New Head

Train with your dataset while the backbone remains fixed.

## Step 6: Fine-Tune if Needed

If validation performance is not enough, unfreeze selected deeper layers and train carefully.

## Key Points

- Start simple: freeze backbone and train the head.
- Fine-tune only if needed.
- Use validation loss to guide decisions.

## Caution Points

- Do not tune using the test set.
- Use smaller learning rates during fine-tuning.
- Watch for overfitting after unfreezing layers.


# Data Preprocessing for Transfer Learning

Pretrained models expect input data in a format similar to their original training setup.

## Common Preprocessing Steps

- Resize images to expected size.
- Convert images to tensors.
- Normalize using expected mean and standard deviation.
- Use augmentations during training.

## Why Normalization Matters

If the pretrained model was trained on normalized images, feeding unnormalized images changes the input distribution.

This can reduce performance because the model receives data unlike what it learned from.

## Common Augmentations

- Random crop
- Horizontal flip
- Color jitter
- Rotation
- Random resized crop

## Key Points

- Preprocessing must match model expectations.
- Augmentation can improve generalization.
- Validation/test transforms should be deterministic.

## Caution Points

- Do not apply random augmentation to validation/test data.
- Do not use unrealistic augmentations.
- Strong augmentation can harm performance if it changes class meaning.


# Famous CNN Architectures in Transfer Learning

## VGG

VGG uses many simple 3 by 3 convolutions.

Strength:

- Simple and easy to understand.

Limitation:

- Large number of parameters.

## ResNet

ResNet uses skip connections.

Skip connections help gradients flow through very deep networks.

Strength:

- Strong and widely used baseline.

## EfficientNet

EfficientNet balances depth, width, and image resolution.

Strength:

- Good accuracy with fewer parameters compared with many older models.

## MobileNet

MobileNet is designed for efficiency.

Strength:

- Useful for mobile and low-resource settings.

## Key Points

- Different architectures trade off speed, memory, and accuracy.
- ResNet is a strong general-purpose choice.
- EfficientNet is often a good efficient choice.
- MobileNet is useful when deployment resources are limited.

## Caution Points

- Bigger architecture is not always better.
- Some models require specific input sizes or preprocessing.
- Choose architecture based on the project constraint, not only leaderboard accuracy.


# Image Segmentation and U-Net

Transfer learning is often used in image classification, but CNN feature learning is also important for segmentation.

## Classification vs Segmentation

Classification asks:

> What is in the image?

Segmentation asks:

> What is in the image, and where is it at the pixel level?

## Why Standard CNNs Need Modification

Classification CNNs compress spatial information:

$$224 \times 224 \rightarrow 7 \times 7 \rightarrow \text{class scores}$$

Segmentation needs output close to the original image size:

$$224 \times 224 \rightarrow 224 \times 224 \text{ mask}$$

## U-Net Idea

U-Net has three main parts:

1. Encoder
2. Decoder
3. Skip connections

## Encoder

The encoder compresses the image and learns high-level features.

It uses convolution and downsampling.

## Decoder

The decoder expands feature maps back to higher resolution.

It uses upsampling or transposed convolution.

## Skip Connections

Skip connections copy spatial details from the encoder to the decoder.

This helps recover fine boundaries.

## Key Points

- U-Net is designed for pixel-level prediction.
- Encoder captures context.
- Decoder restores resolution.
- Skip connections preserve detail.

## Caution Points

- Segmentation masks must align correctly with images.
- Resizing masks requires care; class labels should not be blurred.
- Segmentation uses pixel-wise losses and metrics.


# Evaluating Transfer Learning Models

## Classification Metrics

Use:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix

For imbalanced datasets, accuracy alone can be misleading.

## Segmentation Metrics

### Pixel Accuracy

Measures how many pixels are classified correctly.

### Intersection over Union

IoU compares overlap between predicted mask and true mask:

$$IoU = \frac{\text{Intersection}}{\text{Union}}$$

### Dice Score

Dice measures overlap:

$$Dice = \frac{2 \times \text{Intersection}}{\text{Predicted Area} + \text{True Area}}$$

## Error Analysis

Look at:

- Wrong predictions
- Confident mistakes
- Class-wise performance
- Failure cases under lighting, background, or viewpoint changes

## Key Points

- Always evaluate on unseen data.
- Use metrics that match the task.
- Visual inspection is especially useful in vision.

## Caution Points

- Do not rely only on aggregate metrics.
- A model may learn background shortcuts.
- A good validation score does not remove the need for qualitative inspection.


# Practical Transfer Learning Checklist

## Before Training

- Confirm class names.
- Check number of images per class.
- Inspect sample images.
- Confirm train/validation/test split.
- Use correct image transforms.
- Replace classifier head.

## During Training

- Track training loss.
- Track validation loss.
- Track validation metrics.
- Save the best checkpoint.
- Watch for overfitting.

## Fine-Tuning Decision

Fine-tune if:

- Feature extraction is underfitting.
- Validation performance plateaus.
- Dataset is large enough.

Avoid heavy fine-tuning if:

- Dataset is tiny.
- Validation loss increases quickly.
- Compute is limited.

## Final Key Points

- Transfer learning reuses useful visual knowledge.
- Freeze first, then fine-tune if needed.
- Preprocessing must match the pretrained model.
- Validation data guides architecture and training decisions.
